In [173]:
import requests
import pandas as pd
import math
import folium

from getpass import getpass

In [174]:
# Configurações da API
BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

# Distâncias em metros
RAIO_PARADAS = 500
RAIO_VEICULOS = 3000

# Limite de linhas analisadas
LIMITE_CANDIDATAS = 100

# Sessão de comunicação com a API
sessao_api = requests.Session()

print("Configurações carregadas com sucesso!")

Configurações carregadas com sucesso!


In [175]:
def autenticar():

    token = getpass("Digite seu token SPTrans: ")

    resposta = sessao_api.post(
        f"{BASE_URL}/Login/Autenticar",
        params={"token": token},
        timeout=20
    )

    resposta.raise_for_status()

    if resposta.json() is True:
        print("Autenticação realizada com sucesso!")
        return True

    raise Exception("Não foi possível autenticar na SPTrans.")

In [176]:
autenticar()

Autenticação realizada com sucesso!


True

In [177]:
def calcular_distancia(lat1, lon1, lat2, lon2):

    raio_terra = 6371000

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)

    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    diferenca_lat = lat2 - lat1
    diferenca_lon = lon2 - lon1

    a = (
        math.sin(diferenca_lat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(diferenca_lon / 2) ** 2
    )

    distancia = 2 * raio_terra * math.asin(
        math.sqrt(a)
    )

    return distancia

In [178]:
def consultar_veiculos():

    resposta = sessao_api.get(
        f"{BASE_URL}/Posicao",
        timeout=60
    )

    resposta.raise_for_status()

    dados = resposta.json()

    if not isinstance(dados, dict):
        raise ValueError("Resposta inválida da SPTrans.")

    registros = []

    for linha in dados.get("l") or []:

        for veiculo in linha.get("vs") or []:

            if veiculo.get("py") is None or veiculo.get("px") is None:
                continue

            registros.append({
                "Codigo_linha": linha["cl"],
                "Letreiro": linha["c"],
                "Sentido": linha["sl"],
                "Prefixo": veiculo.get("p"),
                "Latitude": veiculo["py"],
                "Longitude": veiculo["px"],
                "Destino": linha.get("lt0"),
                "Origem": linha.get("lt1")
            })

    return pd.DataFrame(
        registros,
        columns=[
            "Codigo_linha", "Letreiro", "Sentido", "Prefixo",
            "Latitude", "Longitude", "Destino", "Origem"
        ]
    )


# Consultar veículos
df_veiculos = consultar_veiculos()

print("Veículos encontrados:", len(df_veiculos))

display(df_veiculos.head())

Veículos encontrados: 6683


,Codigo_linha,Letreiro,Sentido,Prefixo,Latitude,Longitude,Destino,Origem
0,33635,177Y-10,2,21238,-23.526858,-46.666708,PINHEIROS,METRÔ BARRA FUNDA
1,33635,177Y-10,2,21246,-23.563359,-46.687175,PINHEIROS,METRÔ BARRA FUNDA
2,1629,574W-10,1,55558,-23.557713,-46.571715,METRÔ BELÉM,JD. WALKIRIA
3,1629,574W-10,1,55412,-23.580624,-46.518246,METRÔ BELÉM,JD. WALKIRIA
4,1629,574W-10,1,55237,-23.620384,-46.483513,METRÔ BELÉM,JD. WALKIRIA


In [179]:
# Entrada das coordenadas de origem e destino

#1ª coordenada
#-23.534564
#-46.654302
#-23.525799
#-46.679251

#2ª coordenada
#-23.556840
#-46.600202
#-23.570096
#-46.587572

#Sem linha
#-23.560919
#-46.511362
#-23.537600
#-46.563510


print("SISTEMA DE RECOMENDAÇÃO DE TRANSPORTE")
print("\nInforme as coordenadas geográficas.")

try:
    print("\n--- ORIGEM ---")

    latitude_origem = float(
        input("Latitude da origem: ").strip().replace(",", ".")
    )

    longitude_origem = float(
        input("Longitude da origem: ").strip().replace(",", ".")
    )

    print("\n--- DESTINO ---")

    latitude_destino = float(
        input("Latitude do destino: ").strip().replace(",", ".")
    )

    longitude_destino = float(
        input("Longitude do destino: ").strip().replace(",", ".")
    )

except ValueError:
    raise ValueError(
        "Coordenadas inválidas. Informe valores numéricos."
    )

# Validar os limites geográficos
if not (-90 <= latitude_origem <= 90):
    raise ValueError("Latitude da origem inválida.")

if not (-180 <= longitude_origem <= 180):
    raise ValueError("Longitude da origem inválida.")

if not (-90 <= latitude_destino <= 90):
    raise ValueError("Latitude do destino inválida.")

if not (-180 <= longitude_destino <= 180):
    raise ValueError("Longitude do destino inválida.")

# Organizar as coordenadas
origem = (latitude_origem, longitude_origem)
destino = (latitude_destino, longitude_destino)

if origem == destino:
    raise ValueError("Origem e destino não podem ser iguais.")

print("\nCoordenadas recebidas e validadas!")
print("Origem:", origem)
print("Destino:", destino)

# Calcular distância de cada veículo até a origem
df_veiculos["Distancia_origem"] = df_veiculos.apply(
    lambda v: calcular_distancia(
        v["Latitude"], v["Longitude"],
        origem[0], origem[1]
    ),
    axis=1
)

# Calcular distância de cada veículo até o destino
df_veiculos["Distancia_destino"] = df_veiculos.apply(
    lambda v: calcular_distancia(
        v["Latitude"], v["Longitude"],
        destino[0], destino[1]
    ),
    axis=1
)

# Resumir as informações por linha
df_resumo = df_veiculos.groupby("Codigo_linha").agg(
    Letreiro=("Letreiro", "first"),
    Sentido=("Sentido", "first"),
    Origem=("Origem", "first"),
    Destino=("Destino", "first"),
    Quantidade_veiculos=("Prefixo", "count"),
    Menor_distancia_origem=("Distancia_origem", "min"),
    Menor_distancia_destino=("Distancia_destino", "min")
).reset_index()

# Identificar linhas próximas de pelo menos um dos pontos
df_candidatas = df_resumo[
    (df_resumo["Menor_distancia_origem"] <= RAIO_VEICULOS) |
    (df_resumo["Menor_distancia_destino"] <= RAIO_VEICULOS)
].copy()

# Priorizar linhas com veículos próximos dos dois pontos
df_candidatas["Pontuacao"] = (
    df_candidatas["Menor_distancia_origem"] +
    df_candidatas["Menor_distancia_destino"]
)

df_candidatas = df_candidatas.sort_values("Pontuacao")

# Limitar a quantidade de linhas analisadas
total_candidatas = len(df_candidatas)

df_candidatas = df_candidatas.head(
    LIMITE_CANDIDATAS
).copy()

print("Total de linhas candidatas:", total_candidatas)
print("Linhas selecionadas para análise:", len(df_candidatas))

display(df_candidatas.head(15))

SISTEMA DE RECOMENDAÇÃO DE TRANSPORTE

Informe as coordenadas geográficas.

--- ORIGEM ---

--- DESTINO ---

Coordenadas recebidas e validadas!
Origem: (-23.55684, -46.600202)
Destino: (-23.570096, -46.587572)
Total de linhas candidatas: 174
Linhas selecionadas para análise: 100


,Codigo_linha,Letreiro,Sentido,Origem,Destino,Quantidade_veiculos,Menor_distancia_origem,Menor_distancia_destino,Pontuacao
55,125,3160-10,1,TERM. VL. PRUDENTE,TERM. PQ. D. PEDRO II,4,285.605610,1554.308778,1839.914388
1636,34448,4726-10,2,MOOCA,METRÔ TATUAPÉ,3,1027.909520,922.560545,1950.470065
688,1680,4726-10,1,MOOCA,METRÔ TATUAPÉ,2,1192.629041,772.723439,1965.352480
664,1637,573A-10,1,METRÔ BRESSER,VL. ALPINA,6,32.097584,1936.179636,1968.277220
462,1080,3112-10,1,VL. INDUSTRIAL,METRÔ BELÉM,5,1153.500776,842.727939,1996.228715
1862,35296,2100-21,2,TERM. VL. CARRÃO,TERM. PQ. D. PEDRO II,8,425.043924,1578.015270,2003.059194
1606,34398,573H-10,2,HOSP. SAPOPEMBA,METRÔ BRESSER,5,1291.661747,723.890443,2015.552190
1410,33859,373T-10,2,JD. ITÁPOLIS,METRÔ BRESSER,7,1069.604174,946.134045,2015.738219
1060,33077,311C-10,2,PQ. SÃO LUCAS,BOM RETIRO,4,607.264322,1420.018315,2027.282637
1408,33848,3112-10,2,VL. INDUSTRIAL,METRÔ BELÉM,4,461.082276,1624.491123,2085.573399


In [180]:
# Armazenar as paradas consultadas
cache_paradas = {}

In [181]:
resultados_paradas = []
erros_consulta = []

for _, linha in df_candidatas.iterrows():

    codigo = int(linha["Codigo_linha"])

    try:
        # Consultar paradas da linha
        if codigo not in cache_paradas:

            resposta = sessao_api.get(
                f"{BASE_URL}/Parada/BuscarParadasPorLinha",
                params={"codigoLinha": codigo},
                timeout=30
            )

            # Interromper se a autenticação expirar
            if resposta.status_code == 401:
                print("Autenticação expirada. Renove a sessão.")
                break

            resposta.raise_for_status()

            cache_paradas[codigo] = resposta.json()

        paradas = cache_paradas[codigo]

        if not isinstance(paradas, list) or not paradas:
            continue

        df = pd.DataFrame(paradas)

        df["py"] = pd.to_numeric(df["py"], errors="coerce")
        df["px"] = pd.to_numeric(df["px"], errors="coerce")

        df = df.dropna(subset=["py", "px"])

        if df.empty:
            continue

        # Calcular distâncias até a origem
        df["Distancia_origem"] = df.apply(
            lambda p: calcular_distancia(
                *origem, p["py"], p["px"]
            ),
            axis=1
        )

        # Calcular distâncias até o destino
        df["Distancia_destino"] = df.apply(
            lambda p: calcular_distancia(
                *destino, p["py"], p["px"]
            ),
            axis=1
        )

        # Encontrar as paradas mais próximas
        embarque = df.loc[df["Distancia_origem"].idxmin()]
        desembarque = df.loc[df["Distancia_destino"].idxmin()]

        resultados_paradas.append({
            "Codigo_linha": codigo,
            "Letreiro": linha["Letreiro"],
            "Sentido": linha["Sentido"],
            "Total_paradas": len(df),

            "Embarque": embarque["np"],
            "Codigo_embarque": embarque["cp"],
            "Distancia_embarque": round(
                embarque["Distancia_origem"], 2
            ),

            "Desembarque": desembarque["np"],
            "Codigo_desembarque": desembarque["cp"],
            "Distancia_desembarque": round(
                desembarque["Distancia_destino"], 2
            )
        })

    except requests.exceptions.RequestException as erro:
        erros_consulta.append({
            "Codigo_linha": codigo,
            "Erro": str(erro)
        })


# Organizar os resultados
df_analise_paradas = pd.DataFrame(resultados_paradas)

print("Linhas consultadas:", len(cache_paradas))
print("Linhas com dados válidos:", len(df_analise_paradas))
print("Erros encontrados:", len(erros_consulta))

display(df_analise_paradas)

Linhas consultadas: 100
Linhas com dados válidos: 22
Erros encontrados: 0


,Codigo_linha,Letreiro,Sentido,Total_paradas,Embarque,Codigo_embarque,Distancia_embarque,Desembarque,Codigo_desembarque,Distancia_desembarque
0,125,3160-10,1,5,INHOMERIM B/C,540014153,578.36,JUVENTUS B/C,540014144,409.28
1,1680,4726-10,1,1,GUAIMBE C/B,540014150,1114.21,GUAIMBE C/B,540014150,890.50
2,1637,573A-10,1,7,ORATORIO C/B,540014154,0.00,SAO CRISTOVAO C/B,540014146,0.00
3,1080,3112-10,1,6,INHOMERIM B/C,540014153,578.36,SAO CRISTOVAO B/C,540014147,48.15
4,33859,373T-10,2,7,ORATORIO C/B,540014154,0.00,SAO CRISTOVAO C/B,540014146,0.00
5,33077,311C-10,2,7,ORATORIO C/B,540014154,0.00,SAO CRISTOVAO C/B,540014146,0.00
6,33848,3112-10,2,6,ORATORIO C/B,540014154,0.00,SAO CRISTOVAO C/B,540014146,0.00
7,34405,573A-10,2,6,INHOMERIM B/C,540014153,578.36,SAO CRISTOVAO B/C,540014147,48.15
8,309,311C-10,1,6,INHOMERIM B/C,540014153,578.36,SAO CRISTOVAO B/C,540014147,48.15
9,1091,373T-10,1,6,INHOMERIM B/C,540014153,578.36,SAO CRISTOVAO B/C,540014147,48.15


In [182]:
# Identificar linhas que tenham paradas próximas dos dois pontos.
# Mesmo sem resultados, preservamos as colunas para as próximas etapas.
if df_analise_paradas.empty:
    df_linhas_compativeis = pd.DataFrame(columns=[
        "Codigo_linha", "Letreiro", "Sentido", "Embarque",
        "Codigo_embarque", "Distancia_embarque", "Desembarque",
        "Codigo_desembarque", "Distancia_desembarque",
        "Distancia_caminhada"
    ])
else:
    df_linhas_compativeis = df_analise_paradas.loc[
        (df_analise_paradas["Distancia_embarque"] <= RAIO_PARADAS)
        & (df_analise_paradas["Distancia_desembarque"] <= RAIO_PARADAS)
    ].copy()

    # Soma de distâncias geográficas em linha reta, não percurso pelas ruas.
    df_linhas_compativeis["Distancia_caminhada"] = (
        df_linhas_compativeis["Distancia_embarque"]
        + df_linhas_compativeis["Distancia_desembarque"]
    )

    df_linhas_compativeis = df_linhas_compativeis.sort_values(
        "Distancia_caminhada"
    )

print(
    "Linhas que atendem à origem e ao destino:",
    len(df_linhas_compativeis)
)
display(df_linhas_compativeis)


Linhas que atendem à origem e ao destino: 6


,Codigo_linha,Letreiro,Sentido,Total_paradas,Embarque,Codigo_embarque,Distancia_embarque,Desembarque,Codigo_desembarque,Distancia_desembarque,Distancia_caminhada
2,1637,573A-10,1,7,ORATORIO C/B,540014154,0.0,SAO CRISTOVAO C/B,540014146,0.0,0.0
4,33859,373T-10,2,7,ORATORIO C/B,540014154,0.0,SAO CRISTOVAO C/B,540014146,0.0,0.0
5,33077,311C-10,2,7,ORATORIO C/B,540014154,0.0,SAO CRISTOVAO C/B,540014146,0.0,0.0
6,33848,3112-10,2,6,ORATORIO C/B,540014154,0.0,SAO CRISTOVAO C/B,540014146,0.0,0.0
10,32893,3160-10,2,7,ORATORIO C/B,540014154,0.0,SAO CRISTOVAO C/B,540014146,0.0,0.0
20,33049,373M-10,2,7,ORATORIO C/B,540014154,0.0,SAO CRISTOVAO C/B,540014146,0.0,0.0


In [183]:
import zipfile
import io
from datetime import date

# Endereço do GTFS da SPTrans
URL_GTFS = (
    "https://www.sptrans.com.br/"
    "umbraco/Surface/PerfilDesenvolvedor/BaixarGTFS"
)


def carregar_gtfs():

    resposta = requests.get(
        URL_GTFS,
        timeout=180
    )

    resposta.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resposta.content)) as arquivo:

        arquivos = {
            nome.split("/")[-1].lower(): nome
            for nome in arquivo.namelist()
        }

        necessarios = [
            "routes.txt",
            "trips.txt",
            "stops.txt",
            "stop_times.txt",
            "calendar.txt"
        ]

        for nome in necessarios:
            if nome not in arquivos:
                raise ValueError(
                    f"Arquivo GTFS ausente: {nome}"
                )

        rotas = pd.read_csv(
            arquivo.open(arquivos["routes.txt"]),
            dtype=str
        )

        viagens = pd.read_csv(
            arquivo.open(arquivos["trips.txt"]),
            dtype=str
        )

        paradas = pd.read_csv(
            arquivo.open(arquivos["stops.txt"]),
            dtype=str
        )

        horarios = pd.read_csv(
            arquivo.open(arquivos["stop_times.txt"]),
            dtype=str
        )

        calendario = pd.read_csv(
            arquivo.open(arquivos["calendar.txt"]),
            dtype=str
        )

    return rotas, viagens, paradas, horarios, calendario, resposta.content


# Carregar os dados
(
    df_gtfs_rotas,
    df_gtfs_viagens,
    df_gtfs_paradas,
    df_gtfs_horarios,
    df_gtfs_calendario,
    dados_gtfs
) = carregar_gtfs()

print("Dados GTFS carregados com sucesso!")

Dados GTFS carregados com sucesso!


In [184]:
from datetime import date

def obter_servicos_ativos(data_consulta=None):

    if data_consulta is None:
        data_consulta = date.today()

    dia_semana = [
        "monday", "tuesday", "wednesday",
        "thursday", "friday", "saturday", "sunday"
    ][data_consulta.weekday()]

    calendario = df_gtfs_calendario.copy()

    inicio = pd.to_datetime(
        calendario["start_date"],
        format="%Y%m%d"
    )

    fim = pd.to_datetime(
        calendario["end_date"],
        format="%Y%m%d"
    )

    data = pd.Timestamp(data_consulta)

    servicos = calendario[
        (inicio <= data) &
        (fim >= data) &
        (calendario[dia_semana] == "1")
    ]

    return set(servicos["service_id"])

In [185]:
def validar_trajeto_gtfs(linha, servicos_ativos):

    letreiro = str(linha["Letreiro"]).strip()

    codigo_embarque = str(int(linha["Codigo_embarque"]))
    codigo_desembarque = str(int(linha["Codigo_desembarque"]))

    # Localizar a rota correspondente
    rotas = df_gtfs_rotas[
        df_gtfs_rotas["route_short_name"].astype(str).str.strip()
        == letreiro
    ]

    if rotas.empty:
        return []

    codigos_rotas = rotas["route_id"].tolist()

    # Localizar viagens com serviços ativos
    viagens = df_gtfs_viagens[
        (df_gtfs_viagens["route_id"].isin(codigos_rotas)) &
        (df_gtfs_viagens["service_id"].isin(servicos_ativos))
    ]

    resultados = []

    for _, viagem in viagens.iterrows():

        horarios = df_gtfs_horarios[
            df_gtfs_horarios["trip_id"] == viagem["trip_id"]
        ].copy()

        horarios["stop_sequence"] = pd.to_numeric(
            horarios["stop_sequence"],
            errors="coerce"
        )

        # Encontrar as posições das duas paradas
        sequencias_embarque = horarios[
            horarios["stop_id"] == codigo_embarque
        ]["stop_sequence"].dropna().tolist()

        sequencias_desembarque = horarios[
            horarios["stop_id"] == codigo_desembarque
        ]["stop_sequence"].dropna().tolist()

        # Verificar se existe alguma combinação na ordem correta
        combinacoes = [
            (origem, destino)
            for origem in sequencias_embarque
            for destino in sequencias_desembarque
            if origem < destino
        ]

        if combinacoes:

            seq_embarque, seq_desembarque = min(
                combinacoes,
                key=lambda x: x[1] - x[0]
            )

            resultados.append({
                "Codigo_linha": linha["Codigo_linha"],
                "Letreiro": letreiro,
                "Sentido": linha["Sentido"],
                "Trip_ID": viagem["trip_id"],
                "Service_ID": viagem["service_id"],
                "Direction_ID": viagem["direction_id"],
                "Embarque": linha["Embarque"],
                "Desembarque": linha["Desembarque"],
                "Sequencia_embarque": int(seq_embarque),
                "Sequencia_desembarque": int(seq_desembarque)
            })

    return resultados

In [186]:
# Identificar serviços ativos na data atual
servicos_ativos = obter_servicos_ativos()

resultados_gtfs = []

# Validar cada linha geograficamente compatível
for _, linha in df_linhas_compativeis.iterrows():

    resultado = validar_trajeto_gtfs(
        linha,
        servicos_ativos
    )

    resultados_gtfs.extend(resultado)

# Organizar as viagens aprovadas
df_viagens_compativeis = pd.DataFrame(resultados_gtfs)

if not df_viagens_compativeis.empty:

    print("Viagens diretas encontradas:")
    print(len(df_viagens_compativeis))

    display(df_viagens_compativeis)

else:

    print("Nenhuma viagem direta programada foi encontrada.")

Viagens diretas encontradas:
6


,Codigo_linha,Letreiro,Sentido,Trip_ID,Service_ID,Direction_ID,Embarque,Desembarque,Sequencia_embarque,Sequencia_desembarque
0,1637,573A-10,1,573A-10-0,USD,0,ORATORIO C/B,SAO CRISTOVAO C/B,7,11
1,33859,373T-10,2,373T-10-1,USD,1,ORATORIO C/B,SAO CRISTOVAO C/B,7,11
2,33077,311C-10,2,311C-10-1,USD,1,ORATORIO C/B,SAO CRISTOVAO C/B,27,31
3,33848,3112-10,2,3112-10-1,USD,1,ORATORIO C/B,SAO CRISTOVAO C/B,11,15
4,32893,3160-10,2,3160-10-1,USD,1,ORATORIO C/B,SAO CRISTOVAO C/B,10,14
5,33049,373M-10,2,373M-10-1,USD,1,ORATORIO C/B,SAO CRISTOVAO C/B,19,23


In [187]:
# Quando nenhuma viagem direta é encontrada, manter a execução do notebook.
if df_viagens_compativeis.empty:
    df_opcoes = pd.DataFrame()
    linha_recomendada = None
    print("Nenhuma viagem compatível encontrada.")
else:
    # Relacionar viagens válidas com as distâncias geográficas

    colunas_geograficas = [
        "Codigo_linha",
        "Codigo_embarque",
        "Codigo_desembarque",
        "Distancia_embarque",
        "Distancia_desembarque"
    ]

    df_opcoes = df_viagens_compativeis.merge(
        df_linhas_compativeis[colunas_geograficas],
        on="Codigo_linha",
        how="inner"
    )

    # Calcular distância total de caminhada
    df_opcoes["Distancia_caminhada"] = (
        df_opcoes["Distancia_embarque"] +
        df_opcoes["Distancia_desembarque"]
    )

    # Recuperar quantidade de veículos monitorados
    monitoramento = df_candidatas[
        ["Codigo_linha", "Quantidade_veiculos"]
    ].rename(columns={
        "Quantidade_veiculos": "Veiculos_monitorados"
    })

    df_opcoes = df_opcoes.merge(
        monitoramento,
        on="Codigo_linha",
        how="left"
    )

    # Ordenar pelas prioridades de seleção
    df_opcoes = df_opcoes.sort_values(
        ["Distancia_caminhada", "Veiculos_monitorados"],
        ascending=[True, False]
    )

    # Manter uma viagem representativa de cada linha
    df_opcoes = df_opcoes.drop_duplicates(
        subset=["Codigo_linha"],
        keep="first"
    )

    print("OPÇÕES DE TRANSPORTE DISPONÍVEIS")

    display(
        df_opcoes[
            [
                "Letreiro",
                "Sentido",
                "Embarque",
                "Desembarque",
                "Distancia_caminhada",
                "Veiculos_monitorados",
                "Trip_ID"
            ]
        ]
    )

    # Selecionar a primeira opção
    if not df_opcoes.empty:

        linha_recomendada = df_opcoes.iloc[0].copy()

        print("\nLINHA RECOMENDADA:")
        print(linha_recomendada["Letreiro"])

    else:

        linha_recomendada = None

        print("Nenhuma viagem compatível encontrada.")


OPÇÕES DE TRANSPORTE DISPONÍVEIS


,Letreiro,Sentido,Embarque,Desembarque,Distancia_caminhada,Veiculos_monitorados,Trip_ID
1,373T-10,2,ORATORIO C/B,SAO CRISTOVAO C/B,0.0,7,373T-10-1
0,573A-10,1,ORATORIO C/B,SAO CRISTOVAO C/B,0.0,6,573A-10-0
2,311C-10,2,ORATORIO C/B,SAO CRISTOVAO C/B,0.0,4,311C-10-1
3,3112-10,2,ORATORIO C/B,SAO CRISTOVAO C/B,0.0,4,3112-10-1
4,3160-10,2,ORATORIO C/B,SAO CRISTOVAO C/B,0.0,3,3160-10-1
5,373M-10,2,ORATORIO C/B,SAO CRISTOVAO C/B,0.0,2,373M-10-1



LINHA RECOMENDADA:
373T-10


In [188]:
def converter_horario(horario):
    horas, minutos = map(int, horario.split(":"))
    return horas * 60 + minutos

In [189]:
from IPython.display import display, Markdown


def consultar_previsao(codigo_linha, codigo_parada):

    resposta = sessao_api.get(
        f"{BASE_URL}/Previsao",
        params={
            "codigoLinha": int(codigo_linha),
            "codigoParada": int(codigo_parada)
        },
        timeout=30
    )

    # Renovar autenticação se necessário
    if resposta.status_code == 401:

        print("Renovando autenticação da SPTrans...")

        autenticar()

        resposta = sessao_api.get(
            f"{BASE_URL}/Previsao",
            params={
                "codigoLinha": int(codigo_linha),
                "codigoParada": int(codigo_parada)
            },
            timeout=30
        )

    resposta.raise_for_status()

    dados = resposta.json()
    registros = []

    if isinstance(dados, dict):

        parada = dados.get("p")

        if isinstance(parada, dict):

            for linha in parada.get("l") or []:

                if str(linha.get("cl")) != str(codigo_linha):
                    continue

                for veiculo in linha.get("vs") or []:

                    horario = veiculo.get("t")

                    if not horario:
                        continue

                    espera = None

                    if dados.get("hr"):

                        diferenca = (
                            converter_horario(horario)
                            - converter_horario(dados["hr"])
                        )

                        if diferenca < -720:
                            diferenca += 1440
                        elif diferenca > 720:
                            diferenca -= 1440

                        if diferenca >= 0:
                            espera = diferenca

                    registros.append({
                        "Prefixo": veiculo.get("p"),
                        "Previsao": horario,
                        "Espera_minutos": espera,
                        "Acessivel": veiculo.get("a"),
                        "Referencia": dados.get("hr")
                    })

    return pd.DataFrame(registros)


# ==========================================
# APRESENTAÇÃO DA RECOMENDAÇÃO
# ==========================================

if linha_recomendada is not None:

    linha = linha_recomendada

    distancia_embarque = float(linha["Distancia_embarque"])
    distancia_desembarque = float(linha["Distancia_desembarque"])

    # Apresentar informações do trajeto
    display(Markdown(f"""
# 🚌 Sua recomendação de viagem

**Linha recomendada: {linha['Letreiro']}**

Sentido de operação: {linha['Sentido']}

<|--|>

### 📍 Seu percurso

| Etapa | Localização | Distância em linha reta |
|---|---|---:|
| Origem → Embarque | {linha['Embarque']} | {distancia_embarque:.1f} m |
| Desembarque → Destino | {linha['Desembarque']} | {distancia_desembarque:.1f} m |

**Soma das distâncias em linha reta: {distancia_embarque + distancia_desembarque:.1f} metros**

*Essas medidas não representam necessariamente o percurso real a pé pelas calçadas.*

### ✅ Validação da viagem

- Viagem direta identificada.
- Sequência das paradas confirmada pelo GTFS.
- Viagem programada: `{linha['Trip_ID']}`.
""".replace("<|--|>", "---")))

    # Consultar previsão em tempo real
    try:

        df_previsao_final = consultar_previsao(
            linha["Codigo_linha"],
            linha["Codigo_embarque"]
        )

        if not df_previsao_final.empty:

            df_previsao_final = df_previsao_final.sort_values(
                "Espera_minutos",
                na_position="last"
            )

            proximo = df_previsao_final.iloc[0]

            acessibilidade = (
                "Sim" if proximo["Acessivel"] is True
                else "Não" if proximo["Acessivel"] is False
                else "Não informado"
            )

            espera = (
                f"{int(proximo['Espera_minutos'])} minutos"
                if pd.notna(proximo["Espera_minutos"])
                else "Não foi possível calcular"
            )

            display(Markdown(f"""
### ⏱ Próximo ônibus

| Informação | Resultado |
|---|---|
| Prefixo do veículo | **{proximo['Prefixo']}** |
| Previsão de chegada | **{proximo['Previsao']}** |
| Tempo estimado de espera | **{espera}** |
| Acessível para PcD | **{acessibilidade}** |

<small>Horário de referência da consulta: {proximo['Referencia']}</small>
"""))

            if len(df_previsao_final) > 1:

                print("\nOutros ônibus previstos:")

                display(
                    df_previsao_final.iloc[1:4][
                        [
                            "Prefixo",
                            "Previsao",
                            "Espera_minutos"
                        ]
                    ].rename(columns={
                        "Prefixo": "Veículo",
                        "Previsao": "Chegada prevista",
                        "Espera_minutos": "Espera (min)"
                    }).reset_index(drop=True)
                )

        else:

            display(Markdown("""
### ⏱ Previsão de chegada

A SPTrans não disponibilizou previsões de chegada para este momento.

**A recomendação de viagem continua válida**, pois a sequência programada foi confirmada pelo GTFS.
"""))

    except requests.exceptions.RequestException as erro:

        display(Markdown(f"""
### ⚠️ Previsão temporariamente indisponível

Não foi possível consultar os horários em tempo real.

Detalhe técnico: `{erro}`

A recomendação programada foi preservada.
"""))

else:

    display(Markdown("""
# Não encontramos uma viagem direta

Não foi possível identificar uma linha com todos os critérios necessários para o deslocamento informado.

Experimente outras coordenadas ou um raio de proximidade diferente.
"""))


# 🚌 Sua recomendação de viagem

**Linha recomendada: 373T-10**

Sentido de operação: 2

---

### 📍 Seu percurso

| Etapa | Localização | Distância em linha reta |
|---|---|---:|
| Origem → Embarque | ORATORIO C/B | 0.0 m |
| Desembarque → Destino | SAO CRISTOVAO C/B | 0.0 m |

**Soma das distâncias em linha reta: 0.0 metros**

*Essas medidas não representam necessariamente o percurso real a pé pelas calçadas.*

### ✅ Validação da viagem

- Viagem direta identificada.
- Sequência das paradas confirmada pelo GTFS.
- Viagem programada: `373T-10-1`.



### ⏱ Previsão de chegada

A SPTrans não disponibilizou previsões de chegada para este momento.

**A recomendação de viagem continua válida**, pois a sequência programada foi confirmada pelo GTFS.


In [190]:
# ==========================================
# MAPA INTERATIVO DA RECOMENDAÇÃO
# ==========================================

def gerar_mapa_recomendacao(linha, origem, destino, nome_arquivo="mapa_recomendacao.html"):
    """Gera o mapa, usando projeções sobre os segmentos do traçado GTFS."""

    trip_id = str(linha["Trip_ID"])
    codigo_embarque = str(int(linha["Codigo_embarque"]))
    codigo_desembarque = str(int(linha["Codigo_desembarque"]))

    viagem = df_gtfs_viagens.loc[df_gtfs_viagens["trip_id"] == trip_id]
    if viagem.empty or pd.isna(viagem.iloc[0]["shape_id"]):
        raise ValueError(f"Geometria não identificada para a viagem {trip_id}.")
    shape_id = str(viagem.iloc[0]["shape_id"])

    # A sequência das paradas já foi validada na etapa de recomendação.
    if int(linha["Sequencia_embarque"]) >= int(linha["Sequencia_desembarque"]):
        raise ValueError("Ordem de embarque e desembarque inválida.")

    def coordenadas_parada(codigo):
        parada = df_gtfs_paradas.loc[df_gtfs_paradas["stop_id"] == codigo]
        if parada.empty:
            raise ValueError(f"Parada {codigo} ausente no GTFS.")
        registro = parada.iloc[0]
        return (float(registro["stop_lat"]), float(registro["stop_lon"]))

    embarque = coordenadas_parada(codigo_embarque)
    desembarque = coordenadas_parada(codigo_desembarque)

    # Buscar somente os pontos do itinerário escolhido.
    partes = []
    with zipfile.ZipFile(io.BytesIO(dados_gtfs)) as gtfs:
        nome_shapes = next(
            (nome for nome in gtfs.namelist()
             if nome.split("/")[-1].lower() == "shapes.txt"),
            None
        )
        if nome_shapes is None:
            raise ValueError("shapes.txt não encontrado no GTFS.")

        for bloco in pd.read_csv(
            gtfs.open(nome_shapes),
            usecols=["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
            dtype={"shape_id": str},
            chunksize=200000
        ):
            selecionados = bloco.loc[bloco["shape_id"] == shape_id]
            if not selecionados.empty:
                partes.append(selecionados)

    if not partes:
        raise ValueError(f"Traçado {shape_id} não encontrado.")

    geometria = pd.concat(partes, ignore_index=True)
    for coluna in ["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]:
        geometria[coluna] = pd.to_numeric(geometria[coluna], errors="coerce")
    geometria = geometria.dropna(
        subset=["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"]
    ).sort_values("shape_pt_sequence")

    percurso = list(zip(geometria["shape_pt_lat"], geometria["shape_pt_lon"]))
    if len(percurso) < 2:
        raise ValueError("Geometria GTFS insuficiente.")

    # Para cada segmento, projetar a parada sobre o eixo da linha.
    # Isso é mais preciso do que escolher o vértice mais próximo.
    def candidatos_projecao(parada):
        raio = 6371000
        metros_por_grau_lat = math.pi * raio / 180
        metros_por_grau_lon = (
            metros_por_grau_lat * math.cos(math.radians(parada[0]))
        )
        candidatos = []

        for i in range(len(percurso) - 1):
            a, b = percurso[i], percurso[i + 1]
            ax = (a[1] - parada[1]) * metros_por_grau_lon
            ay = (a[0] - parada[0]) * metros_por_grau_lat
            bx = (b[1] - parada[1]) * metros_por_grau_lon
            by = (b[0] - parada[0]) * metros_por_grau_lat

            dx, dy = bx - ax, by - ay
            comprimento2 = dx * dx + dy * dy

            t = 0 if comprimento2 == 0 else max(
                0, min(1, -(ax * dx + ay * dy) / comprimento2)
            )

            projetado = (
                a[0] + t * (b[0] - a[0]),
                a[1] + t * (b[1] - a[1])
            )
            distancia = calcular_distancia(*parada, *projetado)

            # posicao decimal representa a ordem dentro de shapes.txt
            candidatos.append((distancia, i + t, i, projetado))

        return sorted(candidatos, key=lambda item: item[0])

    # Considerar as projeções mais próximas e exigir ordem de viagem.
    candidatos_embarque = candidatos_projecao(embarque)[:30]
    candidatos_desembarque = candidatos_projecao(desembarque)[:30]

    pares_validos = [
        (e, d)
        for e in candidatos_embarque
        for d in candidatos_desembarque
        if e[1] < d[1]
    ]

    if not pares_validos:
        raise ValueError("Não foi possível recortar o trecho na ordem da viagem.")

    proj_embarque, proj_desembarque = min(
        pares_validos,
        key=lambda par: par[0][0] ** 2 + par[1][0] ** 2
    )

    if max(proj_embarque[0], proj_desembarque[0]) > 150:
        raise ValueError("Parada distante demais do eixo GTFS (mais de 150 m).")

    # Incluir as projeções como extremos, mesmo quando ficam entre vértices.
    i_embarque = proj_embarque[2]
    i_desembarque = proj_desembarque[2]
    ponto_eixo_embarque = proj_embarque[3]
    ponto_eixo_desembarque = proj_desembarque[3]

    trecho_onibus = (
        [ponto_eixo_embarque]
        + percurso[i_embarque + 1:i_desembarque + 1]
        + [ponto_eixo_desembarque]
    )

    # Remover duplicatas consecutivas, caso uma projeção caia em um vértice.
    trecho_onibus_limpo = [trecho_onibus[0]]
    for ponto in trecho_onibus[1:]:
        if calcular_distancia(*trecho_onibus_limpo[-1], *ponto) > 0.01:
            trecho_onibus_limpo.append(ponto)
    trecho_onibus = trecho_onibus_limpo

    mapa = folium.Map(location=embarque, zoom_start=14)

    # Itinerário completo em cinza; trecho usado pelo passageiro em azul.
    folium.PolyLine(
        percurso, color="gray", weight=4, opacity=0.4,
        tooltip="Itinerário completo do GTFS"
    ).add_to(mapa)

    folium.PolyLine(
        trecho_onibus, color="blue", weight=7, opacity=0.9,
        tooltip=f"Trecho de ônibus (GTFS): {trip_id}"
    ).add_to(mapa)

    # Pequenas ligações CARTOGRÁFICAS: a parada fica na lateral da via
    # e o traçado GTFS representa o eixo aproximado. Não são trajetos.
    for parada, ponto_eixo in [
        (embarque, ponto_eixo_embarque),
        (desembarque, ponto_eixo_desembarque)
    ]:
        if calcular_distancia(*parada, *ponto_eixo) > 1:
            folium.PolyLine(
                [parada, ponto_eixo], color="blue", weight=2,
                opacity=0.65, dash_array="2,5",
                tooltip="Ajuste visual entre parada e eixo GTFS (não é rota)"
            ).add_to(mapa)

    # Ligações em linha reta para indicar as distâncias até as paradas.
    # Elas NÃO correspondem a percursos reais pelas calçadas.
    folium.PolyLine(
        [origem, embarque], color="green", weight=3, dash_array="6,8",
        tooltip="Origem → embarque (distância reta, não rota a pé)"
    ).add_to(mapa)
    folium.PolyLine(
        [desembarque, destino], color="red", weight=3, dash_array="6,8",
        tooltip="Desembarque → destino (distância reta, não rota a pé)"
    ).add_to(mapa)

    marcadores = [
        ("Origem informada", origem, "green"),
        (f"Embarque: {linha['Embarque']}", embarque, "blue"),
        (f"Desembarque: {linha['Desembarque']}", desembarque, "purple"),
        ("Destino informado", destino, "red"),
    ]

    for nome, coordenadas, cor in marcadores:
        folium.Marker(
            location=coordenadas, tooltip=nome, popup=nome,
            icon=folium.Icon(color=cor)
        ).add_to(mapa)

    mapa.fit_bounds(trecho_onibus + [origem, destino, embarque, desembarque])
    mapa.save(nome_arquivo)

    print(
        f"Distância parada–eixo GTFS: embarque {proj_embarque[0]:.1f} m; "
        f"desembarque {proj_desembarque[0]:.1f} m."
    )
    return mapa


In [191]:
# Gerar o mapa somente se uma viagem tiver sido recomendada.
if linha_recomendada is not None:
    try:
        mapa_recomendacao = gerar_mapa_recomendacao(
            linha=linha_recomendada,
            origem=origem,
            destino=destino,
            nome_arquivo="mapa_recomendacao.html"
        )
        print("Mapa interativo salvo em: mapa_recomendacao.html")
        print("Abra esse arquivo no navegador para visualizar a viagem.")
        print("Nota: ligações verdes/vermelhas são linhas retas, não rotas reais a pé.")
        print("Conexões azuis finas indicam apenas o ajuste cartográfico parada–eixo GTFS.")
        display(mapa_recomendacao)
    except (ValueError, KeyError, OSError, zipfile.BadZipFile) as erro:
        print(f"O mapa não pôde ser gerado: {erro}")
        print("A recomendação de transporte permanece disponível acima.")
else:
    print("Mapa não gerado: nenhuma viagem direta foi recomendada.")


Distância parada–eixo GTFS: embarque 0.1 m; desembarque 0.7 m.
Mapa interativo salvo em: mapa_recomendacao.html
Abra esse arquivo no navegador para visualizar a viagem.
Nota: ligações verdes/vermelhas são linhas retas, não rotas reais a pé.
Conexões azuis finas indicam apenas o ajuste cartográfico parada–eixo GTFS.
